<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 2C: *Spatial Join Fire Data*

## Metadata
---
### Contents  
> 1. *Set Geometries*
> 2. *Spatial Join of Fire Damage with Main Sampling Grid Data*
> 3. *Export File*
---
### Notes
- Integrate wildfire impact data with daily weather data from gridMET and date from sampling grid. Sampling grid includes topological, social, inrastructure, and land cover data.
---
### Inputs
- Daily Weather Readings - `gridmet_final.csv` 
- Wildfire Damage Data - `fire_data.csv` (cleaned in module 1)
- California Mesh Sampling Grid - `Sampling_grids.csv` (built in appendix A) 
---
### Outputs  
- `samples_projected.csv` Cleaned weather dataset merged with fire damage severity and grid data.
---
### User Created Dependencies  

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Data handling
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime as dt

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point, Polygon

---
## Load Files

In [3]:
fire_data = pd.read_csv("../data/processed/fire_data.csv")
samples = pd.read_csv("../data/processed/samples_res.csv")

## Set Geometry

- All ArcGIS work was performed in EPSG 3310 to preserve area. 

In [ ]:
keep = ['centroid_easting','centroid_northing','Date','grid_id']
join_samples = samples[keep]

join_samples['geometry'] = [Point(xy) for xy in zip(join_samples['centroid_easting'], join_samples['centroid_northing'])]
samples_gdf = gpd.GeoDataFrame(join_samples, geometry='geometry', crs="EPSG:3310")

fire_data['geometry'] = [Point(xy) for xy in zip(fire_data['Fire_Longitude'], fire_data['Fire_Latitude'])]
fire_data_gdf = gpd.GeoDataFrame(fire_data, geometry='geometry', crs="EPSG:4326")

## Project to Equal area projection
fire_data_projected = fire_data_gdf.to_crs(3310)

## Wildfire Grid Intersection

**Main loop of wildfire grid intersect algorithm**
- Loop through each date in fire damage dataset
- Save all fires associated with the current date
    - If no fires, move on to next date
- Load grid centroids associated with the current date
- Create a buffer around each around each fire for current date
    - buffer size is either default 23000 meters radius or scaled up according to acreage burned in meters. (see appendix B)
- Intersection spatial join of grid centroids and buffered fires
- Aggregate in case of multiple fires in a buffered range
    - `Total_estimated_cost` = Total damage in dollars summed for all fires in a grid
    - `Total_fire_count` = Total amount of fires in a grid on the current day
    - `acres` = Total acres burned in a grid on the current day
    - `Damage_missing_flag` = # of flags that signal no data exists for damage, multiple fires are summed up
    - `Acres_missing_flag` = # of flags that signal no data exists for acres, multiple fires are summed up
- Assign samples to new dataframe

In [ ]:
# default buffer radius for square buffer
buffer_size = 23000

In [8]:
for dt in fire_data_projected['Date'].unique():
    
    # Fires on this date
    fires_today = fire_data_projected[fire_data_projected['Date'] == dt]
    if fires_today.empty:
        continue
        
    buffer_size = fires_today['buffer'].max()

    # Samples on this date
    samples_today = samples_gdf[samples_gdf['Date'] == dt]
    if samples_today.empty:
        continue

    # Create buffers around each sample
    fires_buffered = fires_today.copy()
    fires_buffered['geometry'] = fires_buffered['geometry'].apply(lambda p: square_buffer(p, buffer_size))

    # Spatial join: find samples within fire buffers
    joined = gpd.sjoin(samples_today, fires_buffered, how='left', predicate='intersects')

    joined['acres_missing'] = 1 - joined['Acres_has_data']
    joined['damage_missing'] = 1 - joined['Estimated_Damage_has_data']
    
    # Aggregate counts and total damage per sample
    grouped = joined.groupby('grid_id').agg(
        fire_count=('Estimated_Damage', 'count'),
        total_fire_damage=('Estimated_Damage', 'sum'),
        acres = ('Acres', 'sum'),
        fires_missing_acres = ('acres_missing', 'sum'),
        fires_missing_damage=('damage_missing', 'sum'),
    ).fillna(0)

    # Assign values back to main dataframe
    samples_gdf.loc[samples_gdf['Date'] == dt, 'fire_count'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['fire_count']).fillna(0)
    samples_gdf.loc[samples_gdf['Date'] == dt, 'total_fire_damage'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['total_fire_damage']).fillna(0)
    samples_gdf.loc[samples_gdf['Date'] == dt, 'acres'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['acres']).fillna(0)
    samples_gdf.loc[samples_gdf['Date'] == dt, 'fires_missing_acres'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['fires_missing_acres']).fillna(0)
    samples_gdf.loc[samples_gdf['Date'] == dt, 'fires_missing_damage'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['fires_missing_damage']).fillna(0)

## Post Merge Checks

In [9]:
post_merge_check(samples_gdf, samples)

Premerged shape:  (608880, 68)
Merged shape:  (608880, 10)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  30680


In [10]:
samples_gdf.isna().sum()

centroid_easting           0
centroid_northing          0
Date                       0
grid_id                    0
geometry                   0
fire_count              6136
total_fire_damage       6136
acres                   6136
fires_missing_acres     6136
fires_missing_damage    6136
dtype: int64

**Note**: Any NA values are the result of buffered areas that cobntained no fires in range within the date range. Safe to fill these values with 0. 

In [ ]:
samples_gdf = samples_gdf.fillna(0)

# Drop redundant columns
samples_gdf = samples_gdf.drop(columns=['centroid_easting','centroid_northing','geometry'])
samples_gdf

**Merge Temporary fire dataframe with main dataset**

In [14]:
# Merge on BOTH station and date
samples_fires = samples.merge(
    samples_gdf,
    on=['grid_id','Date'],
    how='left'
)

In [15]:
post_merge_check(samples_fires, samples)

Premerged shape:  (608880, 68)
Merged shape:  (608880, 73)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0


NA values after merge:  0


**Sort dataframe by date**

In [16]:
## sort dataframe by date
samples_fires = samples_fires.sort_values(by='Date')

## reset index
samples_fires = samples_fires.reset_index(drop=True)

## Export File

In [17]:
samples_fires.to_csv('../data/processed/samples_fires.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
